In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [ ]:
from abc import ABC, abstractmethod
from pydantic import BaseModel, computed_field
from functools import cached_property
from typing import List, Tuple, Optional, Any, ClassVar, Union
from datasets import load_from_disk


class GenModel(BaseModel, ABC):
    base_model_id: ClassVar[str]

    checkpoint_path: str
    timestamp: Optional[int] = None
    steps: Optional[int] = None
    _model_and_tokenizer: Optional[Tuple[Any, Any]] = None

    def _load_model_and_tokenizer(self) -> Tuple[Any, Any]:
        if self._model_and_tokenizer is None:
            model, tokenizer = self.load_model(self.checkpoint_path)
            self._model_and_tokenizer = (model, tokenizer)
        return self._model_and_tokenizer

    @computed_field
    @cached_property
    def tokenizer(self) -> Any:
        _, tokenizer = self._load_model_and_tokenizer()
        return tokenizer

    @computed_field
    @cached_property
    def gen_model(self) -> Any:
        model, _ = self._load_model_and_tokenizer()
        return model

    @staticmethod
    def load_finetuning_dataset(path, tokenizer, token_func=None, chat=True):
        dataset = load_from_disk(path)

        if token_func is not None:
            dataset = dataset.map(token_func)

        if chat:

            def format_chat_data(example):
                convos = example["text"]
                texts = [
                    tokenizer.apply_chat_template(
                        convo,
                        add_generation_prompt=False,
                        tokenize=False,
                        return_tensors="pt",
                    )
                    for convo in convos
                ]
                return {"text": texts}

            dataset = dataset.map(format_chat_data, batched=True)
        else:

            dataset = dataset.map(
                lambda samples: tokenizer(
                    samples["text"],
                    padding="max_length",
                    truncation=True,
                    max_length=512,
                    add_special_tokens=True,
                ),
            ).shuffle()

        # print first sample tokenized

        return dataset

    @classmethod
    @abstractmethod
    def train_model(
        cls, dataset_path: str, callbacks=None, **kwargs
    ) -> List["GenModel"]:
        """
        Train the model and return a list of fine-tuned checkpoints.

        Args:
            dataset_path: Path to the dataset
            callbacks: Optional list of callbacks to use during training
            **kwargs: Additional training arguments
        """
        pass

    @abstractmethod
    def load_model(self, path: str) -> Any:
        """
        Load the checkpoint from file system.
        """
        pass

    @abstractmethod
    def generate(self, prompt: str, gen_args: dict) -> str:
        """
        Run inference on the given prompt using the fine-tuned model.
        """
        pass

    @abstractmethod
    def resume_training(self, steps: int, callbacks=None) -> List["GenModel"]:
        """
        Resume training from self.

        Args:
            steps: Number of steps to train for
            callbacks: Optional list of callbacks to use during training
        """
        pass


In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from typing import Any, Tuple, ClassVar, List
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from transformers import TrainingArguments

import os
from datetime import datetime


class LLama31GenModel(GenModel):
    base_model_id: ClassVar[str] = "unsloth/Meta-Llama-3.1-8B-bnb-4bit"

    @classmethod
    def train_model(
        cls,
        dataset_path: str,
        epochs: int = 2,
        max_seq_length: int = 512,
        learning_rate: float = 2e-5,
        lr_scheduler: str = "cosine",
        gradient_accumulation_steps: int = 1,
        weight_decay: float = 0.01,
        warmup_steps: int = 100,
        lora_rank: int = 64,
        save_steps: int = 500,
        output_dir: str = "finetuning/sft/models",
    ) -> List["LLama31GenModel"]:
        base_model_id = "unsloth/Meta-Llama-3.1-8B-bnb-4bit"

        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=base_model_id,
            load_in_4bit=True,
            dtype=None,
        )

        # change the padding tokenizer value
        tokenizer.add_special_tokens({"pad_token": "<|reserved_special_token_0|>"})
        model.config.pad_token_id = tokenizer.pad_token_id  # updating model config
        tokenizer.padding_side = (
            "right"  # padding to right (otherwise SFTTrainer shows warning)
        )

        # add eos token at the end of the samples

        def add_eos_token(example):
            example["text"] = example["text"] + tokenizer.eos_token
            return example

        dataset = cls.load_finetuning_dataset(
            path=dataset_path, tokenizer=tokenizer, token_func=add_eos_token
        )
        print(dataset[0])

        response_template = "\n->\n"
        collator = DataCollatorForCompletionOnlyLM(
            tokenizer=tokenizer, response_template=response_template
        )

        model = FastLanguageModel.get_peft_model(
            model,
            r=lora_rank,
            lora_alpha=16,
            lora_dropout=0,
            target_modules=[
                "q_proj",
                "k_proj",
                "v_proj",
                "up_proj",
                "down_proj",
                "o_proj",
                "gate_proj",
            ],
            use_rslora=True,
            use_gradient_checkpointing="unsloth",
        )

        trainer = SFTTrainer(
            model=model,
            train_dataset=dataset,
            tokenizer=tokenizer,
            dataset_text_field="text",
            max_seq_length=max_seq_length,
            data_collator=collator,
            args=TrainingArguments(
                learning_rate=learning_rate,
                lr_scheduler_type=lr_scheduler,
                per_device_train_batch_size=2,
                gradient_accumulation_steps=gradient_accumulation_steps,
                num_train_epochs=epochs,
                fp16=not is_bfloat16_supported(),
                bf16=is_bfloat16_supported(),
                logging_steps=1,
                optim="adamw_8bit",
                weight_decay=weight_decay,
                warmup_steps=warmup_steps,
                output_dir=output_dir,
                save_steps=save_steps,
            ),
        )

        trainer.train()

        timestamp = int(datetime.now().timestamp())
        checkpoint_steps = []
        checkpoint_dir = os.path.join(output_dir)

        if os.path.exists(checkpoint_dir):
            for folder in os.listdir(checkpoint_dir):
                if folder.startswith("checkpoint-"):
                    step = int(folder.split("-")[1])
                    checkpoint_steps.append(step)

        checkpoint_models = []
        for step in checkpoint_steps:
            checkpoint_path = os.path.join(checkpoint_dir, f"checkpoint-{step}")
            model_instance = cls(
                checkpoint_path=checkpoint_path, timestamp=timestamp, steps=step
            )
            checkpoint_models.append(model_instance)

        return checkpoint_models


    def resume_training(self, epochs):
        raise NotImplementedError

    def load_model(self, checkpoint_path: str) -> Tuple[Any, Any]:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=checkpoint_path,
            max_seq_length=512,  # temp fix
            load_in_4bit=True,
            dtype=None,
        )

        tokenizer.add_special_tokens({"pad_token": "<|reserved_special_token_0|>"})
        model.config.pad_token_id = tokenizer.pad_token_id  # updating model config
        tokenizer.padding_side = (
            "right"  # padding to right (otherwise SFTTrainer shows warning)
        )

        model = FastLanguageModel.for_inference(model)

        return model, tokenizer

    def export_gguf(self, checkpoint, quantization_method="fp16"):
        model_folder = f"finetuning/sft/models/{self.base_model_id.split('/')[-1]}/{self.timestamp}"
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=f"{model_folder}/checkpoint-{checkpoint}",
            max_seq_length=self.fine_tuning_arguments.max_seq_length,
            load_in_4bit=True,
            dtype=None,
        )

        tokenizer.add_special_tokens({"pad_token": "<|reserved_special_token_0|>"})
        model.config.pad_token_id = tokenizer.pad_token_id
        tokenizer.padding_side = "right"

        model.save_pretrained_gguf("gguf", tokenizer, quantization_method)

    def generate(self, prompt: str, gen_args: dict) -> str:
        input_ids = self.tokenizer(
            prompt, return_tensors="pt", padding=True, truncation=True
        ).input_ids.to("cuda")
        attention_mask = self.tokenizer(
            prompt, return_tensors="pt", padding=True, truncation=True
        ).attention_mask.to("cuda")

        output = self.gen_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=gen_args.get("max_length", 512),
            do_sample=gen_args.get("do_sample", True),
            temperature=gen_args.get("temperature", 0.7),
            top_p=gen_args.get("top_p", 0.95),
            top_k=gen_args.get("top_k", 50),
        )
        generated_text = self.tokenizer.decode(
            output[0], skip_special_tokens=True, clean_up_tokenization_spaces=True
        )
        return generated_text.strip()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find `steps_per_generation` in grpo_trainer
Unsloth: Could not find `generation_batch_size` in grpo_trainer


In [ ]:
from pydantic import BaseModel
from typing import List, Optional, Dict, Union


# TODO: check whether attachments and URLS fields are always present in datataset
class Prompt(BaseModel):
    subject: str
    attachments: Optional[bool]
    urls: Optional[bool]


class OutputMessage(BaseModel):
    body: str
    # attachments: Optional[List[str]]
    # urls: Optional[List[str]]


class PromptOutputPair(BaseModel):

    prompt: Prompt
    output_message: OutputMessage


def generate_prompt(
    subject: str, attachments: bool = False, urls: bool = False, sentiment:list = ["neutral"]
) -> str:
    prompt = dict()
    prompt["subject"] = subject
    prompt["urls"] = urls
    prompt["attachments"] = attachments

    if sentiment:

        prompt["sentiment"] = ", ".join(sentiment)

    prompt = "\n".join(f"{k}: {v}" for k, v in prompt.items())

    return prompt


def generate_target_value(body):
    target_value = {"body": body}
    target_value = "\n".join(f"{k}: {v}" for k, v in target_value.items())

    return target_value


def generate_prompt_output_pair(
    body: str,
    subject: str,
    attachments: bool = False,
    urls: bool = False,
    sentiment:List[str] = ["neutral"],
) -> str:
    prompt = generate_prompt(subject, attachments, urls, sentiment=sentiment)
    target_value = generate_target_value(body)
    return f"{prompt}\n->\n{target_value}"


In [ ]:
from pydantic import BaseModel
from typing import Optional, List


class MessageGenerator(BaseModel):
    gen_model: GenModel

    def generate_message(
        self,
        subject: str,
        attachments: bool = False,
        sentiment: List[str] = ["neutral"],
        urls: bool = False,
    ):

        prompt = (
            generate_prompt(
                subject=subject, attachments=attachments, urls=urls, sentiment=sentiment
            )
            + "\n->\n"
        )

        gen_args = {
            "max_length": 384,
            "num_return_sequences": 1,
            "top_k": 50,
            "top_p": 0.95,
            "do_sample": True,
            "temperature": 0.9,
        }

        return self.gen_model.generate(
            prompt=prompt,
            gen_args=gen_args,
        )

    def generate_chat_message(
        self,
        subject: str,
        attachments: bool = False,
        sentiment: List[str] = ["neutral"],
        urls: bool = False,
    ):

        messages = [
            {
                "role": "system",
                "content": "You are a helpful assistant that assists in writing emails.\n\nCompose an email based on the features provided by the user.\n\nAll identifiable information should be replaced by corresponding placeholders:\n\nurls -> <URL>\nattachments -> <ATTACHMENT>\nphone numbers -> <PHONE>\ndates -> <DATE>\norganization  -> <ORG>\nemail address -> <EMAIL>\nperson name -> <PER>\naddress/location -> <LOC>",
            },
            {
                "role": "user",
                "content": f"urls: {urls}\nattachments: {attachments}\nsubject: {subject}",
            },
        ]
        gen_args = {
            "max_length": 512,
            "num_return_sequences": 1,
            "top_k": 50,
            "top_p": 0.95,
            "do_sample": True,
            "temperature": 0.75,
        }
        return self.gen_model.generate(prompt=messages, gen_args=gen_args)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
model = LLama31GenModel(checkpoint_path="/content/drive/MyDrive/Thesisproject/Models/checkpoint-2104")

prompt = "Join Us for an Environmental Hackathon: Solve Real-World Challenges"

mess_gen = MessageGenerator(gen_model=model)
response = mess_gen.generate_message(
    subject=prompt,
    attachments=False,
    sentiment=["neutral"],
    urls=True,
)
print(response)


==((====))==  Unsloth 2026.3.18: Fast Llama patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

Unsloth: Will load /content/drive/MyDrive/Thesisproject/Models/checkpoint-2104 as a legacy tokenizer.
Unsloth 2026.3.18 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


subject: Join Us for an Environmental Hackathon: Solve Real-World Challenges
urls: True
attachments: False
sentiment: neutral
->
body: Join Us for an Environmental Hackathon:
Solving Real-World Challenges

<DATE> – 22, <PER>
<ORG>, <LOC>


The environment is a complex arena for policy-makers, businesses and
citizens. It is filled with competing interests, conflicting
information, and the need to make decisions under conditions of deep
uncertainty. A recent survey of federal scientists revealed that they
believe that current scientific information is not adequately used to
inform environmental decisions. This disconnect between science and
policy is a major barrier to improving our environment and reducing
inequality and poverty in the <LOC>.

How can we help bridge this gap between scientific discovery and
real-world application? Join us for a hackathon that brings
together environmental scientists, policymakers, technologists and
other creative thinkers to come up with concrete soluti

Here is where the RL will be applied. Up to now, it is just an architectural model to be written and handled. Ideally, we aim to exploit Reinforcement Learning with Human Feedback (RLHF), an ML Model that exploits human interaction
to get a better, stronger model. The goal, now, is to understand how to include it in the model.
The following code will just provide the ScamLLM model integrated and used to test the email that has been generated in the previous section (that appears with "response"). Then, we stard to have an idea on what and how to do

In [ ]:
import torch
from transformers import pipeline

print("ScamLLM model downloading...")

device = 0 if torch.cuda.is_available() else -1
scam_detector = pipeline(
    task="text-classification",
    model="phishbot/ScamLLM",
    device=device,
    top_k=None # Necessary to retrieve bot Label0 and Label1
)
print("Dowload fine!")

# Reward function
def scam_evasion_reward(prompts, completions, **kwargs):
    """
    Evsluate generated tests and assign  reward.
    prompts: inputs
    completions: tests generated from the RL model
    """
    rewards = []

    # Ensure to extract a pure string for the text
    texts_to_evaluate = []
    for comp in completions:
        # if comp is a list (i.e. [['text']], extract the string)
        if isinstance(comp, list):
             texts_to_evaluate.append(str(comp[0]))
        else:
             texts_to_evaluate.append(str(comp))

    # truncation=True to avoid crashes if LLaMA generates a text longer than 512 characters
    results = scam_detector(texts_to_evaluate, truncation=True, max_length=512)

    for res in results:
        scores = {item['label']: item['score'] for item in res}

        safe_score = scores.get('LABEL_0', 0.0) # LABEL_0 = Safe, LABEL_1 = Malicious
        rewards.append(safe_score)

    return rewards

"""
If you use our model in your research, please cite our paper "From Chatbots to Phishbots?: Phishing Scam Generation in Commercial Large Language Models" (https://www.computer.org/csdl/proceedings-article/sp/2024/313000a221/1WPcYLpYFHy).

BibTeX below:

  title={From Chatbots to Phishbots?: Phishing Scam Generation in Commercial Large Language Models},
  author={Roy, Sayak Saha and Thota, Poojitha and Naragam, Krishna Vamsi and Nilizadeh, Shirin},
  booktitle={2024 IEEE Symposium on Security and Privacy (SP)},
  pages={221--221},
  year={2024},
  organization={IEEE Computer Society}
}
"""

ScamLLM model downloading...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Dowload fine!


'\nIf you use our model in your research, please cite our paper "From Chatbots to Phishbots?: Phishing Scam Generation in Commercial Large Language Models" (https://www.computer.org/csdl/proceedings-article/sp/2024/313000a221/1WPcYLpYFHy).\n\nBibTeX below:\n\n  title={From Chatbots to Phishbots?: Phishing Scam Generation in Commercial Large Language Models},\n  author={Roy, Sayak Saha and Thota, Poojitha and Naragam, Krishna Vamsi and Nilizadeh, Shirin},\n  booktitle={2024 IEEE Symposium on Security and Privacy (SP)},\n  pages={221--221},\n  year={2024},\n  organization={IEEE Computer Society}\n}\n'

EXPLOIT GENERATION + DETECTION TO CREATE A SUITED DATABASE WITH A SET OF "EMAIL - SAFE PERCENTAGE" COLUMNS

In [ ]:
# --- Text model's output safe percentage ---
print("--- Test model output's safe percentage ---")

test_prompts = [prompt]
test_completions = [response]

response_rewards = scam_evasion_reward(prompts=test_prompts, completions=test_completions)

for text, reward in zip(test_completions, response_rewards):
    print(f"\nText:\n{text}")
    print(f"{'-'*40}")
    print(f"Safe percentage: {reward:.4f} ({reward*100:.2f}%)")

--- Test model output's safe percentage ---

Text:
subject: Join Us for an Environmental Hackathon: Solve Real-World Challenges
urls: True
attachments: False
sentiment: neutral
->
body: Join Us for an Environmental Hackathon:
Solving Real-World Challenges

<DATE> – 22, <PER>
<ORG>, <LOC>


The environment is a complex arena for policy-makers, businesses and
citizens. It is filled with competing interests, conflicting
information, and the need to make decisions under conditions of deep
uncertainty. A recent survey of federal scientists revealed that they
believe that current scientific information is not adequately used to
inform environmental decisions. This disconnect between science and
policy is a major barrier to improving our environment and reducing
inequality and poverty in the <LOC>.

How can we help bridge this gap between scientific discovery and
real-world application? Join us for a hackathon that brings
together environmental scientists, policymakers, technologists and
othe

In [ ]:
import pandas as pd
import json
import os

# Write some prompt examples; ideally, try to write
# as much malicious prompt as safe ones
prompts = [
    # malicious. With high percentage, the result will be labeled as malicious
    {"subject": "URGENT: Your account will be suspended in 24 hours.", "sentiment": ["urgent", "threatening"], "urls": True, "attachments": False},
    {"subject": "Security Alert: Unusual login attempt blocked", "sentiment": ["urgent", "neutral"], "urls": True, "attachments": False},
    {"subject": "Invoice #84920 payment overdue - Action Required", "sentiment": ["formal", "urgent"], "urls": True, "attachments": True},

    # safe. With high percentage, the result will be labeled as safe
    {"subject": "Quarterly Employee Recognition: Your €100 Retail Gift Card", "sentiment": ["professional", "joyful"], "urls": False, "attachments": False},
    {"subject": "Join Us for an Environmental Hackathon: Solve Real-World Challenges", "sentiment": ["neutral", "friendly"], "urls": True, "attachments": False},
    {"subject": "IT Update: Scheduled Maintenance for Internal Servers", "sentiment": ["informative", "formal"], "urls": False, "attachments": False}
]

# prompts = prompts * 3 # If we desire multiple emails with the same starting prompt

dataset_records = []
threshold = 0.50 # Recall: safe when > 50%

print(f"Dataset Generation: {len(prompts)} emails...\n")

for idx, prompt in enumerate(prompts):
    print(f"Generating {idx+1}/{len(prompts)}: '{prompt['subject']}'")

    # Message generation
    output_text = mess_gen.generate_message(
        subject=prompt["subject"],
        attachments=prompt["attachments"],
        sentiment=prompt["sentiment"],
        urls=prompt["urls"],
    )

    if "->\n" in output_text:
        prompt_part, completion_part = output_text.split("->\n", 1)
        prompt_part += "->\n"
    else:
        prompt_part = f"subject: {prompt['subject']}\nurls: {prompt['urls']}\nattachments: {prompt['attachments']}\nsentiment: {', '.join(prompt['sentiment'])}\n->\n"
        completion_part = output_text

    completion_part = completion_part.strip()

    eval_text = completion_part

    reward = scam_evasion_reward(prompts=[""], completions=[eval_text])[0]

    # T/F evaluation for KTO/BCO
    is_safe = bool(reward >= threshold)

    print(f"  Score ScamLLM: {reward:.4f} ({reward*100:.2f}%) -> Label KTO/BCO: {is_safe}\n")

    dataset_records.append({
        "prompt": prompt_part.strip(),
        "completion": completion_part,
        "label": is_safe
    })

# Saving the dataset as a jsonl
df = pd.DataFrame(dataset_records)

dataset_dir = "/content/drive/MyDrive/Thesisproject/Dataset"
os.makedirs(dataset_dir, exist_ok=True)

save_path = os.path.join(dataset_dir, "empirical_dataset_kto_bco.jsonl")
df.to_json(save_path, orient="records", lines=True)

print("="*60)
print(f"Dataset succesfully generated with: {len(df)} elements")
print(f"Saved in: {save_path}")
print("="*60)


# Forcing pandas to show all the rows and not truncate the columns text
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

display(df)

# Reset default pandas options for future calls (if needed)
pd.reset_option('display.max_rows')
pd.reset_option('display.max_colwidth')

Dataset Generation: 6 emails...

Generating 1/6: 'URGENT: Your account will be suspended in 24 hours.'
  Score ScamLLM: 0.0012 (0.12%) -> Label KTO/BCO: False

Generating 2/6: 'Security Alert: Unusual login attempt blocked'
  Score ScamLLM: 0.5602 (56.02%) -> Label KTO/BCO: True

Generating 3/6: 'Invoice #84920 payment overdue - Action Required'
  Score ScamLLM: 0.0032 (0.32%) -> Label KTO/BCO: False

Generating 4/6: 'Quarterly Employee Recognition: Your €100 Retail Gift Card'
  Score ScamLLM: 0.0849 (8.49%) -> Label KTO/BCO: False

Generating 5/6: 'Join Us for an Environmental Hackathon: Solve Real-World Challenges'
  Score ScamLLM: 0.0186 (1.86%) -> Label KTO/BCO: False

Generating 6/6: 'IT Update: Scheduled Maintenance for Internal Servers'
  Score ScamLLM: 0.9507 (95.07%) -> Label KTO/BCO: True

Dataset succesfully generated with: 6 elements
Saved in: /content/drive/MyDrive/Thesisproject/Dataset/empirical_dataset_kto_bco.jsonl


,prompt,completion,label
0,"subject: URGENT: Your account will be suspended in 24 hours.\nurls: True\nattachments: False\nsentiment: urgent, threatening\n->","body: Important information for you about the terms of your account\n\n<DATE> 23:00 GMT (17:00 Pacific time)\n\nDear <PER>,\n\nWe need to send you some important information about your <ORG> account.\n\nYour account will be suspended in 24 hours for failing to comply with our User Agreement.\n\nIf you have not been in touch with us within 48 hours, your account will be suspended. \n\nTo avoid having your account suspended, click here to read the information that we need from you. <URL>\n\nIf you have questions, please contact Customer Service at <PHONE>.\n\nThanks,\n\n<ORG> Customer Service\n\nPlease note: The e-mail address you see in your from line may differ from the one you provided when you opened your account. This e-mail was sent to <EMAIL>.\n\nWe hope you enjoyed receiving this message. However, if you'd rather not receive future e-mails of this sort from <ORG>, please visit the Help page Updating subscriptions and communication preferences. <URL> Please note that this e-mail was sent to the following e-mail address: <EMAIL>\n\n(c) 2002 <ORG> & Co., 1600 Powell Street, Palo Alto, CA 94304. <ORG> and the <ORG> logo are registered trademarks of <ORG>.",False
1,"subject: Security Alert: Unusual login attempt blocked\nurls: True\nattachments: False\nsentiment: urgent, neutral\n->","body: Dear Community Member:\n\nIn the past few hours we have received an unusually high number of login attempts from computers that have failed our standard security checks. This may be the result of a computer virus or other online attack. As a precautionary measure we have temporarily blocked logins from new IP addresses. You will be unable to access our website using a new IP address unless you have an administrator reset your password using the instructions on our website<<URL> If you experience this problem, please contact us using our website contact form<<URL> We apologize for the inconvenience, and are working to restore normal service as quickly as possible.\n\nPlease contact us if you have any questions or concerns.\n\nSincerely,\nThe <ORG> Community Team",True
2,"subject: Invoice #84920 payment overdue - Action Required\nurls: True\nattachments: True\nsentiment: formal, urgent\n->","body: Please find attached invoice for payment. This invoice is over 30 days\nprior to due date and payment has not yet been received by <ORG>.\nPayment is required immediately to avoid late payment charges and other\nconsequences. Please note that all invoices must be paid in <LOC> Dollars\nand that <ORG> is not obligated to pay invoices submitted in any other\ncurrency.\n\nIf you have any questions in regards to this invoice or any other invoice,\nplease contact John K. Buss, Director of Accounts Payable at <PHONE>\nor <EMAIL> <EMAIL>>.\n\nSincerely,\n\nJohn K. Buss\nDirector of Accounts Payable\n\n<ATTACHMENT>\n\n\n\n<ATTACHMENT>",False
3,"subject: Quarterly Employee Recognition: Your €100 Retail Gift Card\nurls: False\nattachments: False\nsentiment: professional, joyful\n->","body: Quarterly Employee Recognition\n\nYou have just been invited to attend the Quarterly Employee Recognition in <LOC> on <DATE>!\n\nEvent Details\nLocation: <LOC>\nDate: <DATE>\nTime: 11:30AM - 1:30PM\n\nIn attendance will be <PER>, <PER>, <PER> and your <LOC> management team.\n\nWhat to Expect\nYou will have the opportunity to mingle and network with the <LOC> management team and hear from our special guest speaker, <PER>. You will also have the chance to find out more about <ORG>'s business and what we are doing in <LOC>. The highlight of the event will be the announcement of the winning employees for the <ORG> Recognition Program. The winning employees will have received the highest rating from their supervisor during the most recent quarterly performance review. These employees will be recognized on stage and will r

KTO TRAINER

In [ ]:
import torch
import gc
from datasets import load_dataset
from unsloth import FastLanguageModel, PatchFastRL, is_bfloat16_supported
from trl import KTOConfig, KTOTrainer

PatchFastRL("KTO", FastLanguageModel)
torch.cuda.empty_cache()
gc.collect()

dataset_path = "/content/drive/MyDrive/Thesisproject/Dataset/empirical_dataset_kto_bco.jsonl"
sft_model_path = "/content/drive/MyDrive/Thesisproject/Models/checkpoint-2104"
output_dir_kto = "/content/drive/MyDrive/Thesisproject/Datasets/empirical_kto"

print("Upload initial model for KTO...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=sft_model_path,
    max_seq_length=512,
    load_in_4bit=True,
    fast_inference=False,
)

FastLanguageModel.for_training(model)

tokenizer.add_special_tokens({"pad_token": "<|reserved_special_token_0|>"})
model.config.pad_token_id = tokenizer.pad_token_id
tokenizer.padding_side = "right"

print("Upload the dataset previously created...")
train_dataset = load_dataset("json", data_files=dataset_path, split="train")

print("KTO Configuration (8-bit Low-VRAM)...")
kto_args = KTOConfig(
    output_dir=output_dir_kto,
    learning_rate=5e-6,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,

    # Memory management
    optim="adamw_8bit",            # Uses less VRAM GIGABYTE w.r.t. standard AdamW
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    gradient_checkpointing=True,

    max_prompt_length=128,
    max_completion_length=384,

    num_train_epochs=3,
    logging_steps=1,
    beta=0.1,
    remove_unused_columns=False,
)

kto_trainer = KTOTrainer(
    model=model,
    ref_model=None,
    args=kto_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
)

# Patch for transformers 5.0
kto_trainer._get_train_sampler = lambda *args, **kwargs: super(type(kto_trainer), kto_trainer)._get_train_sampler()

print("Start KTO training...")
kto_trainer.train()
kto_trainer.save_model(output_dir_kto)
print("KTO Model saved in: ", output_dir_kto)

Unsloth: UnslothAlignPropTrainer is already patched.
Unsloth: UnslothBCOTrainer is already patched.
Unsloth: UnslothCPOTrainer is already patched.
Unsloth: UnslothDDPOTrainer is already patched.
Unsloth: UnslothDPOTrainer is already patched.
Unsloth: UnslothGKDTrainer is already patched.
Unsloth: UnslothGRPOTrainer is already patched.
Unsloth: UnslothKTOTrainer is already patched.
Unsloth: UnslothNashMDTrainer is already patched.
Unsloth: UnslothOnlineDPOTrainer is already patched.
Unsloth: UnslothORPOTrainer is already patched.
Unsloth: UnslothPPOTrainer is already patched.
Unsloth: UnslothPRMTrainer is already patched.
Unsloth: UnslothRewardTrainer is already patched.
Unsloth: UnslothRLOOTrainer is already patched.
Unsloth: UnslothSFTTrainer is already patched.
Unsloth: UnslothXPOTrainer is already patched.
Upload initial model for KTO...
==((====))==  Unsloth 2026.3.18: Fast Llama patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: L

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load /content/drive/MyDrive/Thesisproject/Models/checkpoint-2104 as a legacy tokenizer.
Unsloth 2026.3.18 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Upload the dataset previously created...
KTO Configuration (8-bit Low-VRAM)...


/content/unsloth_compiled_cache/UnslothKTOTrainer.py:1123: UserWarning: You have different amounts of desirable/positive and undesirable/negative examples but the weights on the desirable and undesirable losses don't seem to be in an ideal range. Based on your data, we recommend EITHER desirable_weight in [2.0, 2.66] or undesirable_weight in [0.38, 0.5] (but NOT BOTH). See the documentation on how to optimally set these weights.
  warnings.warn(


Start KTO training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6 | Num Epochs = 3 | Total steps = 6
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,kl,logits / chosen,logits / rejected,rewards / chosen,rewards / margins,rewards / rejected
1,0.749226,180.088150,-19049972.000000,-40165970.666667,10.636424,-9.829857,20.466281
2,0.995423,212.774567,-59625840.000000,-6388501.500000,15.895429,-10.764056,26.659485
3,0.749131,170.082611,-19125920.000000,-39966749.333333,10.648566,-8.479595,19.128161
4,0.981254,223.950714,-60453196.000000,-6525814.000000,18.437225,-7.915693,26.352919
5,0.748972,154.092728,-19220034.000000,-39629989.333333,10.575806,-6.444623,17.020429
6,0.949994,231.729568,-61036512.000000,-6548496.500000,20.228653,-5.888609,26.117262


KTO Model saved in:  /content/drive/MyDrive/Thesisproject/Datasets/empirical_kto


BCO Trainer

In [ ]:
import torch
import gc
from datasets import load_dataset
from unsloth import FastLanguageModel, PatchFastRL
from trl import BCOConfig, BCOTrainer

PatchFastRL("BCO", FastLanguageModel)
torch.cuda.empty_cache()
gc.collect()

dataset_path = "/content/drive/MyDrive/Thesisproject/Dataset/empirical_dataset_kto_bco.jsonl"
sft_model_path = "/content/drive/MyDrive/Thesisproject/Models/checkpoint-2104"
output_dir_bco = "/content/drive/MyDrive/Thesisproject/Datasets/empirical_bco"

print("Upload initial model for BCO...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=sft_model_path,
    max_seq_length=512,
    load_in_4bit=True,
    fast_inference=False,
)

FastLanguageModel.for_training(model)
tokenizer.add_special_tokens({"pad_token": "<|reserved_special_token_0|>"})
model.config.pad_token_id = tokenizer.pad_token_id
tokenizer.padding_side = "right"

print("Upload the dataset previously created...")
train_dataset = load_dataset("json", data_files=dataset_path, split="train")

print("BCO Configuration...")
bco_args = BCOConfig(
    output_dir=output_dir_bco,
    learning_rate=5e-6,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    logging_steps=1,
    beta=0.1,
    max_prompt_length=256,
    max_completion_length=512,
    remove_unused_columns=False,
)

bco_trainer = BCOTrainer(
    model=model,
    ref_model=None,
    args=bco_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
)

bco_trainer._get_train_sampler = lambda *args, **kwargs: super(type(bco_trainer), bco_trainer)._get_train_sampler()

print("Start BCO training...")
bco_trainer.train()
bco_trainer.save_model(output_dir_bco)
print("BCO Model saved in: ", output_dir_bco)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find `steps_per_generation` in grpo_trainer
Unsloth: Could not find `generation_batch_size` in grpo_trainer
Unsloth: UnslothAlignPropTrainer is already patched.
Unsloth: UnslothBCOTrainer is already patched.
Unsloth: UnslothCPOTrainer is already patched.
Unsloth: UnslothDDPOTrainer is already patched.
Unsloth: UnslothDPOTrainer is already patched.
Unsloth: UnslothGKDTrainer is already patched.
Unsloth: UnslothGRPOTrainer is already patched.
Unsloth: UnslothKTOTrainer is already patched.
Unsloth: UnslothNashMDTrainer is already patched.
Unsloth: UnslothOnlineDPOTrainer is already patched.
Unsloth: UnslothORPOTrainer is already patched.
Unsloth: UnslothPPOTrainer is already patched.
Unsloth: UnslothPRMTrainer is already patched.
Unsloth: UnslothRewardTrainer is already patched.
Unsloth: UnslothRLOOTrainer is already patched.
Unsloth:

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load /content/drive/MyDrive/Thesisproject/Models/checkpoint-2104 as a legacy tokenizer.
Unsloth 2026.3.18 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Upload the dataset previously created...
BCO Configuration...


Map (num_proc=6):   0%|          | 0/6 [00:00<?, ? examples/s]

...

Filtering undesirable examples (num_proc=6):   0%|          | 0/6 [00:00<?, ? examples/s]

Start BCO training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6 | Num Epochs = 3 | Total steps = 6
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss,delta,logits / chosen,logits / rejected,rewards / chosen,rewards / margins,rewards / rejected
1,3.335527,21.744724,-59625840.000000,-32343992.000000,15.895429,-8.422084,24.317513
2,4.242654,19.099218,-32420352.000000,-31251036.000000,10.632553,-4.482239,15.114792
3,4.130125,19.994053,-59625840.000000,-32343992.000000,15.895429,-8.422084,24.317513
4,4.242654,19.099218,-32420352.000000,-31251036.000000,10.632553,-4.482239,15.114792
5,4.353393,19.643902,-59625840.000000,-32343992.000000,15.895429,-8.422084,24.317513
6,3.610539,19.169523,-32832718.000000,-31253706.000000,11.965270,-3.082296,15.047566


BCO Model saved in:  /content/drive/MyDrive/Thesisproject/Datasets/empirical_bco


Evaluate the models

In [ ]:
import torch
import gc

torch.cuda.empty_cache()
gc.collect()

model_to_train = "kto"
#model_to_train = "bco"
# Due to memory reasons, cannot test both of them in the same execution

if model_to_train == "kto":
    model_path = "/content/drive/MyDrive/Thesisproject/Datasets/empirical_kto"
else:
    model_path = "/content/drive/MyDrive/Thesisproject/Datasets/empirical_bco"

print(f"Uploading model {model_to_train.upper()}...")
model = LLama31GenModel(checkpoint_path=model_path)
mess_gen_test = MessageGenerator(gen_model=model)

# Suppose we want to test the model with a "malicious" prompt
test_subject = "Mandatory Security Upgrade: Verify your identity to keep your account active"
test_sentiment = ["formal", "urgent"]
test_urls = True

print(f"\ncEmail Generation ({model_to_train.upper()})...")
email_output = mess_gen_test.generate_message(
    subject=test_subject,
    attachments=False,
    sentiment=test_sentiment,
    urls=test_urls,
)

# Parsing
if "->\n" in email_output:
    header, body = email_output.split("->\n", 1)
else:
    header = f"subject: {test_subject}\nurls: {test_urls}\nattachments: False\nsentiment: {', '.join(test_sentiment)}\n->\n"
    body = email_output

print("="*75)
print(f"Email generated starting from {model_to_train.upper()}")
print("="*75)
print(body.strip())
print("="*75)

print("\n ScamLLM is evaluating...")
try:
    eval_text = header + body
    rewards = scam_evasion_reward(prompts=[""], completions=[eval_text])
    score = rewards[0]

    print(f"Score for {model_to_train.upper()}: {score*100:.2f}%\n")
except Exception as e:
    print(f"Evaluation error: {e}. Check if ScamLLM has been correctly uploaded")

Uploading model KTO...

cEmail Generation (KTO)...
==((====))==  Unsloth 2026.3.18: Fast Llama patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-bnb-4bit as a legacy tokenizer.


Email generated starting from KTO
body: To keep your account active, you must verify your identity. Please verify your identity now. 

If you do not verify your identity, your account will be suspended and you will be unable to view your account information or use your credit, debit, or prepaid card. 

Verify your identity now.  

Click or tap here to start.  

<URL>

If you have been using a bookmark or saved link to access your account via a web browser, please delete your current bookmark or saved link and use the URL shown above to access your account in the future. 

If you need further assistance, please contact your financial institution.

Thank you. 

The <ORG> Support Team

 ScamLLM is evaluating...
Score for KTO: 1.98%



text evaluation integration

In [ ]:
# pip install -U sentence-transformers
from sentence_transformers import SentenceTransformer, util

# Upload evaluation
sim_model = SentenceTransformer('all-MiniLM-L6-v2')

# MINILM: Deep Self-Attention Distillation for
# Task-Agnostic Compression of Pre-Trained Transformers
# https://arxiv.org/pdf/2002.10957

text_1 = "Subject: Urgent Security Update. Please verify your identity."
text_2 = "To keep your account active, please click the link below to verify your ID."
text_3 = "The HR department is happy to give you a $100 gift card."

# Compute embeddingd
embeddings = sim_model.encode([text_1, text_2, text_3])

# Compute cosine similarity. https://arxiv.org/pdf/1908.10084
choherent_sim = util.cos_sim(embeddings[0], embeddings[1])
wrong_sim = util.cos_sim(embeddings[0], embeddings[2])
perfect_sim = util.cos_sim(embeddings[0], embeddings[0])

print(f"Coherency Security - Verify ID: {choherent_sim[0][0]:.4f}")
print(f"Coherency Security - Gift Card: {wrong_sim[0][0]:.4f}")
print(f"Coherency Security - Security (should be 100% as compare a text with himself): {perfect_sim[0][0]:.4f}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Coherency Security - Verify ID: 0.5656
Coherency Security - Gift Card: 0.2045
Coherency Security - Security (should be 100% as compare a text with himself): 1.0000


In [ ]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, util

print("Upload SBERT model (all-MiniLM-L6-v2)...")
device = "cuda" if torch.cuda.is_available() else "cpu"
sim_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

example = [
    {
        "prompt": "Write an urgent email about a suspended account and ask to click a link.",
        "sft_output": "Your account is suspended! Click this link immediately to restore it: <URL>",
        "kto_output": "To comply with security policies, please update your account using this link to avoid suspension: <URL>",
    }
]
#TODO: Instead of exampe, use the JSON
df = pd.DataFrame(example)

print("Model uploaded. Starting similarity computation...\n")

df['Sim_Prompt_vs_SFT'] = 0.0
df['Sim_SFT_vs_KTO'] = 0.0
df['Sim_Prompt_vs_KTO'] = 0.0

for index, row in df.iterrows():
    embeddings = sim_model.encode([
        row['prompt'],
        row['sft_output'],
        row['kto_output']
    ], convert_to_tensor=True)

    emb_prompt = embeddings[0]
    emb_sft = embeddings[1]
    emb_kto = embeddings[2]


    sim_prompt_sft = util.cos_sim(emb_prompt, emb_sft).item() * 100
    sim_sft_kto = util.cos_sim(emb_sft, emb_kto).item() * 100
    sim_prompt_kto = util.cos_sim(emb_prompt, emb_kto).item() * 100

    df.at[index, 'Sim_Prompt_vs_SFT'] = sim_prompt_sft
    df.at[index, 'Sim_SFT_vs_KTO'] = sim_sft_kto
    df.at[index, 'Sim_Prompt_vs_KTO'] = sim_prompt_kto


print("RESULTS SBERT COHERENCE ANALYSIS")
print("=" * 60)
for index, row in df.iterrows():
    print(f"Prompt: {row['prompt']}")
    print("-" * 60)
    print(f"1. (Prompt vs SFT) : {row['Sim_Prompt_vs_SFT']:.2f}%")
    print(f"2. (SFT vs KTO)    : {row['Sim_SFT_vs_KTO']:.2f}%")
    print(f"3. (Prompt vs KTO) : {row['Sim_Prompt_vs_KTO']:.2f}%")
    print("=" * 60)

⏳ Caricamento del modello SBERT (all-MiniLM-L6-v2)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model uploaded. Starting similarity computation...

RESULTS SBERT COHERENCE ANALYSIS
Prompt: Write an urgent email about a suspended account and ask to click a link.
------------------------------------------------------------
1. (Prompt vs SFT) : 56.41%
2. (SFT vs KTO)    : 71.73%
3. (Prompt vs KTO) : 59.59%


In [ ]:
from unsloth import FastLanguageModel

import torch
import gc
import pandas as pd
import os

torch.cuda.empty_cache()
gc.collect()

# =====================================================================
# 6 PROMPT (3 Malicious, 3 Safe)
# =====================================================================
prompts_data = [
    {"subject": "URGENT: Your account will be suspended in 24 hours.", "sentiment": ["urgent", "threatening"], "urls": True, "attachments": False},
    {"subject": "Security Alert: Unusual login attempt blocked", "sentiment": ["urgent", "neutral"], "urls": True, "attachments": False},
    {"subject": "Invoice #84920 payment overdue - Action Required", "sentiment": ["formal", "urgent"], "urls": True, "attachments": True},
    {"subject": "Quarterly Employee Recognition: Your €100 Retail Gift Card", "sentiment": ["professional", "joyful"], "urls": False, "attachments": False},
    {"subject": "Join Us for an Environmental Hackathon: Solve Real-World Challenges", "sentiment": ["neutral", "friendly"], "urls": True, "attachments": False},
    {"subject": "IT Update: Scheduled Maintenance for Internal Servers", "sentiment": ["informative", "formal"], "urls": False, "attachments": False}
]


def generate_per_model(model_name, model_path, prompts):
    print(f"\nUpload {model_name.upper()} in VRAM...")
    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_path,
            max_seq_length=512,
            load_in_4bit=True,
            fast_inference=False # <--- FIX: Motore standard (niente errore vLLM)
        )
        FastLanguageModel.for_inference(model)
    except Exception as e:
        print(f"Error while uploading {model_name}: {e}")
        return ["ERROR"] * len(prompts)

    test_results = []

    print(f"Generating {len(prompts)} emails with {model_name.upper()}...")
    for idx, p in enumerate(prompts):
        prompt_string = f"subject: {p['subject']}\nurls: {p['urls']}\nattachments: {p['attachments']}\nsentiment: {', '.join(p['sentiment'])}\n->\nbody: "

        inputs = tokenizer([prompt_string], return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True, temperature=0.7)
        generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

        if "->\nbody:" in generated_text:
            body = generated_text.split("->\nbody:")[1].strip()
        else:
            body = generated_text.strip()

        test_results.append(body)
        print(f"Email {idx+1}/{len(prompts)} completed.")

    print(f"Cleaning VRAM from {model_name.upper()}...")
    del model
    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()

    return test_results

path_sft = "/content/drive/MyDrive/Thesisproject/Models/checkpoint-2104"
path_kto = "/content/drive/MyDrive/Thesisproject/Datasets/empirical_kto"
path_bco = "/content/drive/MyDrive/Thesisproject/Datasets/empirical_bco"

prompt_texts = [f"subject: {p['subject']}\nsentiment: {', '.join(p['sentiment'])}" for p in prompts_data]

# Built example dictionary
dati_master = {
    "Original_prompt": prompt_texts,
    "SFT_Output": generate_per_model("SFT_Base", path_sft, prompts_data),
    "KTO_Output": generate_per_model("KTO", path_kto, prompts_data),
    "BCO_Output": generate_per_model("BCO", path_bco, prompts_data)
}



df_master = pd.DataFrame(dati_master)

dataset_dir = "/content/drive/MyDrive/Thesisproject/Dataset"
os.makedirs(dataset_dir, exist_ok=True)
save_path = os.path.join(dataset_dir, "master_evaluation_dataset.jsonl")

df_master.to_json(save_path, orient="records", lines=True)

print("\n" + "="*80)
print(f"MASTER DATASET GENERATED")
print(f"Saved in: {save_path}")
print("="*80)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
display(df_master.head(2))
pd.reset_option('display.max_colwidth')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find `steps_per_generation` in grpo_trainer
Unsloth: Could not find `generation_batch_size` in grpo_trainer

Upload SFT_BASE in VRAM...
==((====))==  Unsloth 2026.3.18: Fast Llama patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load /content/drive/MyDrive/Thesisproject/Models/checkpoint-2104 as a legacy tokenizer.
Unsloth 2026.3.18 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Generating 6 emails with SFT_BASE...
Email 1/6 completed.
Email 2/6 completed.
Email 3/6 completed.
Email 4/6 completed.
Email 5/6 completed.
Email 6/6 completed.
Cleaning VRAM from SFT_BASE...

Upload KTO in VRAM...
==((====))==  Unsloth 2026.3.18: Fast Llama patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

Generating 6 emails with KTO...
Email 1/6 completed.
Email 2/6 completed.
Email 3/6 completed.
Email 4/6 completed.
Email 5/6 completed.
Email 6/6 completed.
Cleaning VRAM from KTO...

Upload BCO in VRAM...
==((====))==  Unsloth 2026.3.18: Fast Llama patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-bnb-4bit as a legacy tokenizer.


Generating 6 emails with BCO...
Email 1/6 completed.
Email 2/6 completed.
Email 3/6 completed.
Email 4/6 completed.
Email 5/6 completed.
Email 6/6 completed.
Cleaning VRAM from BCO...

MASTER DATASET GENERATED
Saved in: /content/drive/MyDrive/Thesisproject/Dataset/master_evaluation_dataset.jsonl


,Original_prompt,SFT_Output,KTO_Output,BCO_Output
0,subject: URGENT: Your account will be suspende...,24 HOURS TO ACTIVATE YOUR ACCOUNT\n\nYour acco...,24 HOURS WARNING: Your Account Will Be Suspend...,24 HOURS WARNING\nYOU ARE ABOUT TO BE SUSPENDE...
1,subject: Security Alert: Unusual login attempt...,2016-10-07 12:14:35 -- Security Alert\n\nSomeo...,202 - Unusual login attempt blocked\n\nSomeone...,12:31:38 AM <DATE>\n\nUnusual login attempt bl...


In [ ]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, util

# 1. Upload Master Dataset
file_path = "/content/drive/MyDrive/Thesisproject/Dataset/master_evaluation_dataset.jsonl"
print(f"Upload dataset from path: {file_path}")
df = pd.read_json(file_path, lines=True)

# 2. Upload SBERT
print("Upload SBERT model (all-MiniLM-L6-v2)...")
device = "cuda" if torch.cuda.is_available() else "cpu"
sim_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

print("Start computing coherency metrics...\n")

results = []

for index, row in df.iterrows():
    prompt = row['Original_prompt']
    sft = row['SFT_Output']
    kto = row['KTO_Output']
    bco = row['BCO_Output']

    subject_label = prompt.split('\n')[0].replace('subject: ', '').strip()

    # 3. Computed vectorial embeddings
    embeddings = sim_model.encode([prompt, sft, kto, bco], convert_to_tensor=True)
    emb_prompt, emb_sft, emb_kto, emb_bco = embeddings[0], embeddings[1], embeddings[2], embeddings[3]

    # 4. Compute similarity percentages
    sim_prompt_sft = util.cos_sim(emb_prompt, emb_sft).item() * 100

    sim_sft_kto = util.cos_sim(emb_sft, emb_kto).item() * 100
    sim_prompt_kto = util.cos_sim(emb_prompt, emb_kto).item() * 100

    sim_sft_bco = util.cos_sim(emb_sft, emb_bco).item() * 100
    sim_prompt_bco = util.cos_sim(emb_prompt, emb_bco).item() * 100


    results.append({
        "Scenario": subject_label,
        "SFT_Baseline_Coherence": sim_prompt_sft,
        "KTO_Semantic_Drift": sim_sft_kto,
        "KTO_Final_Coherence": sim_prompt_kto,
        "BCO_Semantic_Drift": sim_sft_bco,
        "BCO_Final_Coherence": sim_prompt_bco
    })

# Create final DataFrame
df_results = pd.DataFrame(results)

# Saving in CSV
results_path = "/content/drive/MyDrive/Thesisproject/Dataset/coherence_metrics_results.csv"
df_results.to_csv(results_path, index=False)


print("SBERT COHERENCE RESULTS ANALYSIS on avg")
print("=" * 60)
print(f"1. Baseline Coherence (Prompt vs SFT) : {df_results['SFT_Baseline_Coherence'].mean():.2f}%")
print("-" * 60)
print(f"2. KTO Semantic Drift (SFT vs KTO)    : {df_results['KTO_Semantic_Drift'].mean():.2f}%")
print(f"3. KTO Final Coherence (Prompt vs KTO): {df_results['KTO_Final_Coherence'].mean():.2f}%")
print("-" * 60)
print(f"4. BCO Semantic Drift (SFT vs BCO)    : {df_results['BCO_Semantic_Drift'].mean():.2f}%")
print(f"5. BCO Final Coherence (Prompt vs BCO): {df_results['BCO_Final_Coherence'].mean():.2f}%")
print("=" * 60)
print(f"Results saved in: {results_path}")
print("\nFurther details for each scenario:")

display(df_results.round(2))

Upload dataset from path: /content/drive/MyDrive/Thesisproject/Dataset/master_evaluation_dataset.jsonl
Upload SBERT model (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Start computing coherency metrics...

SBERT COHERENCE RESULTS ANALYSIS on avg
1. Baseline Coherence (Prompt vs SFT) : 57.42%
------------------------------------------------------------
2. KTO Semantic Drift (SFT vs KTO)    : 60.03%
3. KTO Final Coherence (Prompt vs KTO): 54.40%
------------------------------------------------------------
4. BCO Semantic Drift (SFT vs BCO)    : 61.93%
5. BCO Final Coherence (Prompt vs BCO): 59.61%
Results saved in: /content/drive/MyDrive/Thesisproject/Dataset/coherence_metrics_results.csv

Further details for each scenario:


,Scenario,SFT_Baseline_Coherence,KTO_Semantic_Drift,KTO_Final_Coherence,BCO_Semantic_Drift,BCO_Final_Coherence
0,URGENT: Your account will be suspended in 24 h...,62.32,71.25,61.58,64.81,65.72
1,Security Alert: Unusual login attempt blocked,53.45,55.18,67.01,62.45,65.29
2,Invoice #84920 payment overdue - Action Required,57.02,17.07,15.51,59.29,57.99
3,Quarterly Employee Recognition: Your €100 Reta...,45.84,69.99,56.88,59.86,55.17
4,Join Us for an Environmental Hackathon: Solve ...,74.10,74.33,65.83,77.77,59.85
5,IT Update: Scheduled Maintenance for Internal ...,51.76,72.39,59.57,47.41,53.63


seems kto wins